# 05.01 Training Data Setup — Patient-Grouped Leakage-Free Splits

Creates patient-disjoint **Track A (Authentic-only)** and **Track B (Synthetic-augmented training, Authentic-only validation/test)** partitions using canonical `patient_group_id` (`charity_patient_no` + fallback).

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_SEED = 42

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "tabular" / "samples" / "charity-patient.csv"
POLICY_JSON = PROJECT_ROOT / "models" / "phase_05" / "feature_exclusion_policy.json"
MAPPINGS_JSON = PROJECT_ROOT / "data" / "interim" / "cleaned" / "category_mappings.json"
PROC_TAB_DIR = PROJECT_ROOT / "data" / "processed" / "tabular"
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
REPORTS_DIR = PROJECT_ROOT / "reports" / "phase_05"

for d in (PROC_TAB_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

policy = json.loads(POLICY_JSON.read_text(encoding="utf-8"))
NUM_COLS = policy["numerical_features"]
CAT_COLS = policy["categorical_features"]
FEATURE_COLS = policy["approved_features"]

df_raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"Loaded raw dataset (read-only): {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")


Loaded raw dataset (read-only): 1000 rows x 71 columns


In [2]:
def parse_currency(val):
    if pd.isna(val):
        return np.nan
    s = str(val).replace(" ", "").replace("₱", "").replace("PHP", "").replace("Php", "")
    m = re.search(r"[\d,]+(?:\.\d+)?", s)
    if not m:
        return np.nan
    try:
        return float(m.group(0).replace(",", ""))
    except ValueError:
        return np.nan

df = df_raw.copy()

# 1. Parse all numerical / currency columns to float64
for col in NUM_COLS:
    if col in df.columns:
        df[col] = df[col].apply(parse_currency)

# 2. Apply canonical taxonomy mappings where available
if MAPPINGS_JSON.exists():
    mappings = json.loads(MAPPINGS_JSON.read_text(encoding="utf-8"))
    for col in ["residence_status", "marital_status", "educational_attainment", "housing_condition"]:
        if col in df.columns and col in mappings:
            df[col] = df[col].replace(mappings[col])

# 3. Canonical patient identity key (never exposed to X)
fallback_key = (
    df["surname"].fillna("").astype(str).str.strip().str.upper()
    + "|" + df["first_name"].fillna("").astype(str).str.strip().str.upper()
    + "|" + df["birthday"].fillna("").astype(str).str.strip().str.upper()
)
df["patient_group_id"] = np.where(
    df["charity_patient_no"].notna() & (df["charity_patient_no"].astype(str).str.strip() != ""),
    df["charity_patient_no"].astype(str).str.strip().str.upper(),
    "FALLBACK_" + fallback_key,
)

# 4. Quarantine unadjudicated PENDING / NaN eligibility rows
adj_mask = df["eligibility"].isin(["ELIGIBLE", "INELIGIBLE"])
df_pending = df.loc[~adj_mask].copy()
df_adj = df.loc[adj_mask].copy().reset_index(drop=True)
df_adj["target"] = (df_adj["eligibility"] == "ELIGIBLE").astype(int)

# 5. Separate AUTHENTIC (deduplicated to 1 record per unique patient) vs SYNTHETIC
df_auth_all = df_adj[df_adj["data_source"] == "AUTHENTIC"].copy()
df_auth = df_auth_all.drop_duplicates(subset=["patient_group_id"], keep="first").reset_index(drop=True)
df_syn = df_adj[df_adj["data_source"] == "SYNTHETIC"].copy().reset_index(drop=True)

print(f"Quarantined unadjudicated/PENDING rows: {len(df_pending)}")
print(f"Authentic adjudicated rows: {len(df_auth_all)} -> Unique Authentic patients: {len(df_auth)}")
print(f"Unique Authentic class balance: {df_auth['eligibility'].value_counts().to_dict()}")
print(f"Synthetic adjudicated rows: {len(df_syn)} (across {df_syn['patient_group_id'].nunique()} parent patients)")

# Save derived datasets
df_auth.to_csv(PROC_TAB_DIR / "ofphil_authentic_patients.csv", index=False)
df_syn.to_csv(PROC_TAB_DIR / "ofphil_synthetic_augmentation.csv", index=False)


Quarantined unadjudicated/PENDING rows: 10
Authentic adjudicated rows: 150 -> Unique Authentic patients: 119
Unique Authentic class balance: {'ELIGIBLE': 75, 'INELIGIBLE': 44}
Synthetic adjudicated rows: 840 (across 117 parent patients)


In [3]:
# 6. Patient-disjoint 80/20 Train-Dev / Final Holdout Test Split on AUTHENTIC patients ONLY
sgkf_holdout = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
auth_train_idx, auth_test_idx = next(
    sgkf_holdout.split(df_auth[FEATURE_COLS], df_auth["target"], groups=df_auth["patient_group_id"])
)

train_patients = set(df_auth.iloc[auth_train_idx]["patient_group_id"])
test_patients = set(df_auth.iloc[auth_test_idx]["patient_group_id"])
assert len(train_patients & test_patients) == 0, "Patient leaked across Authentic train/test!"

# 7. Track B: Synthetic rows whose authentic parent is EXCLUSIVELY in train_patients
syn_train_mask = df_syn["patient_group_id"].isin(train_patients)
syn_train_idx = np.where(syn_train_mask)[0]
syn_train_patients = set(df_syn.iloc[syn_train_idx]["patient_group_id"])
assert len(syn_train_patients & test_patients) == 0, "Synthetic clone of test patient leaked into training!"

# 8. Outer 10-Fold StratifiedGroupKFold on Authentic Train-Dev patients
df_auth_tr = df_auth.iloc[auth_train_idx].reset_index(drop=True)
sgkf_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_folds = []
for f_i, (tr_i, va_i) in enumerate(
    sgkf_cv.split(df_auth_tr[FEATURE_COLS], df_auth_tr["target"], groups=df_auth_tr["patient_group_id"]),
    start=1,
):
    tr_pts = set(df_auth_tr.iloc[tr_i]["patient_group_id"])
    va_pts = set(df_auth_tr.iloc[va_i]["patient_group_id"])
    assert len(tr_pts & va_pts) == 0, f"Patient leaked in CV fold {f_i}!"
    cv_folds.append({"fold": f_i, "train_indices": tr_i.tolist(), "val_indices": va_i.tolist()})

# Verify zero prohibited columns in FEATURE_COLS
PROHIBITED = {
    "charity_patient_no", "surname", "first_name", "middle_name", "contact_no",
    "address", "birthday", "philhealth_no", "data_source", "eligibility", "status",
    "approved_financial_assistance", "remaining_medical_expenses",
    "social_worker_assessment_rating", "relevant_social_worker_findings",
    "assessment_notes", "social_economic_assessment", "case_study_narrative",
}
assert len(PROHIBITED & set(FEATURE_COLS)) == 0, f"Prohibited features found: {PROHIBITED & set(FEATURE_COLS)}"

split_manifest = {
    "dataset_authentic": str((PROC_TAB_DIR / "ofphil_authentic_patients.csv").relative_to(PROJECT_ROOT)),
    "dataset_synthetic": str((PROC_TAB_DIR / "ofphil_synthetic_augmentation.csv").relative_to(PROJECT_ROOT)),
    "n_authentic_patients": int(len(df_auth)),
    "n_authentic_train_dev": int(len(auth_train_idx)),
    "n_authentic_test": int(len(auth_test_idx)),
    "n_synthetic_train_eligible": int(len(syn_train_idx)),
    "n_features": int(len(FEATURE_COLS)),
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
    "feature_cols": FEATURE_COLS,
    "auth_train_indices": auth_train_idx.tolist(),
    "auth_test_indices": auth_test_idx.tolist(),
    "syn_train_indices": syn_train_idx.tolist(),
    "cv_folds": cv_folds,
    "random_seed": RANDOM_SEED,
    "zero_patient_overlap_verified": True,
    "zero_prohibited_features_verified": True,
}
(MODELS_DIR / "phase05_split_and_schema.json").write_text(json.dumps(split_manifest, indent=2), encoding="utf-8")

prov_report = {
    "raw_rows": int(len(df_raw)),
    "quarantined_pending_rows": int(len(df_pending)),
    "authentic_adjudicated_rows": int(len(df_auth_all)),
    "authentic_unique_patients": int(len(df_auth)),
    "authentic_train_dev_patients": int(len(auth_train_idx)),
    "authentic_train_dev_class_counts": df_auth.iloc[auth_train_idx]["eligibility"].value_counts().to_dict(),
    "authentic_test_patients": int(len(auth_test_idx)),
    "authentic_test_class_counts": df_auth.iloc[auth_test_idx]["eligibility"].value_counts().to_dict(),
    "synthetic_total_rows": int(len(df_syn)),
    "synthetic_train_augmentation_rows": int(len(syn_train_idx)),
    "synthetic_excluded_test_clones": int(len(df_syn) - len(syn_train_idx)),
}
(REPORTS_DIR / "dataset_provenance_report.json").write_text(json.dumps(prov_report, indent=2), encoding="utf-8")
print(json.dumps(prov_report, indent=2))
print("Phase 05 patient-grouped split & schema verified and exported.")


{
  "raw_rows": 1000,
  "quarantined_pending_rows": 10,
  "authentic_adjudicated_rows": 150,
  "authentic_unique_patients": 119,
  "authentic_train_dev_patients": 95,
  "authentic_train_dev_class_counts": {
    "ELIGIBLE": 60,
    "INELIGIBLE": 35
  },
  "authentic_test_patients": 24,
  "authentic_test_class_counts": {
    "ELIGIBLE": 15,
    "INELIGIBLE": 9
  },
  "synthetic_total_rows": 840,
  "synthetic_train_augmentation_rows": 688,
  "synthetic_excluded_test_clones": 152
}
Phase 05 patient-grouped split & schema verified and exported.
